# Data Warehouse ETL

Dit notebook laadt data van SDM naar DWH en detecteert eerst welk DWH-schema actief is in `BikeToDrive_6_Datawarehouse.db`:
- **Oud schema**: alle dimensies en feiten gebruiken natural keys
- **Nieuw schema**: SCD Type 1 voor vaste dimensies en SCD Type 2 voor Klant en Monteur
- **Feiten**: Accessoire_Verkoop, Fiets_Verkoop, Onderhoud

**SDM-inlaadstrategie**: het SDM-notebook gebruikt een **full refresh**: eerst alle SDM-tabellen leegmaken, daarna alles opnieuw inladen.

**Let op**: draai dit notebook van boven naar beneden. Dimensies eerst, dan feiten.

In [33]:
# 1. IMPORTS EN INSTELLINGEN
import sys
import sqlite3
import pandas as pd
from pathlib import Path
from loguru import logger

log_path = Path("../logs/dwh_load.log")
log_path.parent.mkdir(parents=True, exist_ok=True)
LOG_FORMAT = "{time:YYYY-MM-DD HH:mm:ss}|{level}|{message}"

logger.remove()
logger.add(
    sys.stderr,
    level="INFO",
    format=LOG_FORMAT,
)
logger.add(
    log_path,
    level="INFO",
    format=LOG_FORMAT,
    encoding="utf-8",
    backtrace=True,
    diagnose=False,
)

DB_SDM = "../WEEK 3/BikeToDrive_6 _SourceDataModel.db"
DB_DWH = "BikeToDrive_6_Datawarehouse.db"
VANDAAG = pd.Timestamp.now().strftime("%Y-%m-%d")

logger.info(f"DWH_ETL|Initialize databases: SDM={DB_SDM}, DWH={DB_DWH}")

2026-04-22 00:38:57|INFO|DWH_ETL|Initialize databases: SDM=../WEEK 3/BikeToDrive_6 _SourceDataModel.db, DWH=BikeToDrive_6_Datawarehouse.db


In [34]:
# 2. ETL CONFIGURATIE
# Detecteer eerst welk DWH-schema actief is in de huidige database.

def detect_dwh_strategy(db_path):
    with sqlite3.connect(db_path) as conn:
        klant_cols = {row[1] for row in conn.execute("PRAGMA table_info(Klant)")}
        monteur_cols = {row[1] for row in conn.execute("PRAGMA table_info(Monteur)")}
        accessoire_verkoop_cols = {
            row[1] for row in conn.execute("PRAGMA table_info(Accessoire_Verkoop)")
        }

    uses_scd2 = (
        {"klant_sk", "begintijd", "eindtijd"}.issubset(klant_cols)
        and {"monteur_sk", "begintijd", "eindtijd"}.issubset(monteur_cols)
        and "klant_sk" in accessoire_verkoop_cols
        and "monteur_sk" in accessoire_verkoop_cols
    )
    return "scd2" if uses_scd2 else "scd1"


DWH_STRATEGIE = detect_dwh_strategy(DB_DWH)

BASE_SCD1_DIMS = [
    ("Datum", "datum", """
        SELECT datum FROM Accessoire_Verkoop
        UNION SELECT datum FROM Fiets_Verkoop
        UNION SELECT datum FROM Onderhoud
    """),
    ("Filiaal", "filiaalnr", """
        SELECT filiaalnr, naam, adres, provincie FROM Accessoire_Verkoop_Filiaal
        UNION SELECT filiaalnr, naam, adres, provincie FROM Fiets_Verkoop_Filiaal
        UNION SELECT filiaalnr, naam, adres, provincie FROM Onderhoud_Filiaal
    """),
    ("Leverancier", "leveranciernr", """
        SELECT leveranciernr, naam, adres, woonplaats FROM Accessoire_Verkoop_Leverancier
        UNION SELECT leveranciernr, naam, adres, woonplaats FROM Accessoire_Inkoop_Leverancier
    """),
    ("Accessoire", "accessoirenr", """
        SELECT accessoirenr, naam, soort, standaardprijs, inkoopprijs FROM Accessoire_Verkoop_Accessoire
        UNION SELECT accessoirenr, naam, soort, standaardprijs, inkoopprijs FROM Accessoire_Inkoop_Accessoire
    """),
    ("Fabrikant", "fabrikantnr", """
        SELECT fabrikantnr, naam, adres, plaats FROM Fiets_Verkoop_Fabrikant
        UNION SELECT fabrikantnr, naam, adres, plaats FROM Onderhoud_Fabrikant
        UNION SELECT fabrikantnr, naam, adres, plaats FROM Fiets_Inkoop_Fabrikant
    """),
    ("Fiets", "fietsnr", """
        SELECT fietsnr, soort, merk, type, kleur, standaardprijs, inkoopprijs FROM Fiets_Verkoop_Fiets
        UNION SELECT fietsnr, soort, merk, type, kleur, standaardprijs, inkoopprijs FROM Onderhoud_Fiets
        UNION SELECT fietsnr, soort, merk, type, kleur, standaardprijs, inkoopprijs FROM Fiets_Inkoop_Fiets
    """),
]

OPTIONELE_SCD1_DIMS = [
    ("Klant", "klantnr", """
        SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Accessoire_Verkoop_Klant
        UNION SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Fiets_Verkoop_Klant
    """),
    ("Monteur", "monteurnr", """
        SELECT monteurnr, naam, woonplaats, uurloon FROM Accessoire_Verkoop_Monteur
        UNION SELECT monteurnr, naam, woonplaats, uurloon FROM Fiets_Verkoop_Monteur
        UNION SELECT monteurnr, naam, woonplaats, uurloon FROM Onderhoud_Monteur
    """),
]

SCD2_DIMS = [
    ("Klant", "klantnr", """
        SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Accessoire_Verkoop_Klant
        UNION SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Fiets_Verkoop_Klant
    """),
    ("Monteur", "monteurnr", """
        SELECT monteurnr, naam, woonplaats, uurloon FROM Accessoire_Verkoop_Monteur
        UNION SELECT monteurnr, naam, woonplaats, uurloon FROM Fiets_Verkoop_Monteur
        UNION SELECT monteurnr, naam, woonplaats, uurloon FROM Onderhoud_Monteur
    """),
] if DWH_STRATEGIE == "scd2" else []

SCD1_DIMS = BASE_SCD1_DIMS + ([] if DWH_STRATEGIE == "scd2" else OPTIONELE_SCD1_DIMS)

FEITEN = [
    ("Accessoire_Verkoop", "accessoire_verkoopnr", """
        SELECT av.accessoire_verkoopnr, av.datum,
               av.klant AS klantnr, av.monteur AS monteurnr,
               m.filiaal AS filiaalnr, av.accessoire AS accessoirenr,
               a.leverancier AS leveranciernr, av.aantal,
               a.standaardprijs, av.verkoopprijs, a.inkoopprijs
        FROM Accessoire_Verkoop av
        JOIN Accessoire_Verkoop_Accessoire a ON av.accessoire = a.accessoirenr
        JOIN Accessoire_Verkoop_Monteur m ON av.monteur = m.monteurnr
    """),
    ("Fiets_Verkoop", "fiets_verkoopnr", """
        SELECT fv.fiets_verkoopnr, fv.datum,
               fv.klant AS klantnr, fv.monteur AS monteurnr,
               m.filiaal AS filiaalnr, fv.fiets AS fietsnr,
               f.fabrikant AS fabrikantnr, fv.aantal,
               f.standaardprijs, fv.verkoopprijs, f.inkoopprijs
        FROM Fiets_Verkoop fv
        JOIN Fiets_Verkoop_Fiets f ON fv.fiets = f.fietsnr
        JOIN Fiets_Verkoop_Monteur m ON fv.monteur = m.monteurnr
    """),
    ("Onderhoud", "onderhoudnr", """ 
        SELECT o.onderhoudnr, o.datum,
               o.monteur AS monteurnr, m.filiaal AS filiaalnr,
               o.fiets AS fietsnr, f.fabrikant AS fabrikantnr,
               o.starttijd, o.eindtijd, m.uurloon
        FROM Onderhoud o
        JOIN Onderhoud_Fiets f ON o.fiets = f.fietsnr
        JOIN Onderhoud_Monteur m ON o.monteur = m.monteurnr
    """),
]

ALLE_TABELLEN = [t for t, _, _ in SCD1_DIMS] + [t for t, _, _ in SCD2_DIMS] + [t for t, _, _ in FEITEN]

logger.info(f"DWH-strategie gedetecteerd: {DWH_STRATEGIE.upper()}")
logger.info(f"Config: {len(SCD1_DIMS)} SCD1, {len(SCD2_DIMS)} SCD2, {len(FEITEN)} feiten")

2026-04-22 00:39:00|INFO|DWH-strategie gedetecteerd: SCD2
2026-04-22 00:39:00|INFO|Config: 6 SCD1, 2 SCD2, 3 feiten


In [35]:
# 3. GENERIEKE FUNCTIES

def get_table_columns(conn, tabel):
    return {row[1] for row in conn.execute(f"PRAGMA table_info({tabel})")}


def afleiden(df, tabel):
    # Voeg afgeleide kolommen toe per dimensie en verwijder ruwe meetwaarden
    if tabel == "Datum":
        dt = pd.to_datetime(df["datum"])
        df["jaar"], df["maand"] = dt.dt.year, dt.dt.month
        df["kwartaal"] = (dt.dt.month - 1) // 3 + 1
        df["datum"] = dt.dt.strftime("%Y-%m-%d")
    elif tabel == "Klant":
        leeftijd = (pd.Timestamp.now() - pd.to_datetime(df["geboortedatum"])).dt.days // 365
        df["leeftijdsklasse"] = pd.cut(
            leeftijd,
            bins=[0, 18, 30, 50, 65, 200],
            labels=["Jong", "Jongvolwassen", "Middelbaar", "Senior", "65+"],
        ).astype(str)
    elif tabel == "Monteur":
        df["loonklasse"] = pd.cut(
            df["uurloon"],
            bins=[0, 15, 25, 35, float("inf")],
            labels=["Laag", "Midden", "Hoog", "Top"],
        ).astype(str)
        df = df.drop(columns=["uurloon"])
    elif tabel in ("Accessoire", "Fiets"):
        bins = [0, 10, 25, 50, float("inf")] if tabel == "Accessoire" else [0, 500, 1000, 2000, float("inf")]
        df["prijsklasse"] = pd.cut(
            df["standaardprijs"],
            bins=bins,
            labels=["Budget", "Midden", "Premium", "Luxe"],
        ).astype(str)
        df = df.drop(columns=["standaardprijs", "inkoopprijs"])
    return df


def vergelijk(df_src, df_dwh, pk):
    # Detecteer nieuwe en gewijzigde rijen t.o.v. DWH
    if df_dwh.empty:
        return df_src.copy(), pd.DataFrame(columns=df_src.columns)

    nieuwe = df_src[~df_src[pk].isin(df_dwh[pk])].copy()
    bestaand = df_src[df_src[pk].isin(df_dwh[pk])]
    if bestaand.empty:
        return nieuwe, pd.DataFrame(columns=df_src.columns)

    merged = bestaand.merge(df_dwh, on=pk, suffixes=("_src", "_dwh"))
    data_cols = [c for c in df_src.columns if c != pk]
    verschilt = pd.Series(False, index=merged.index)
    for col in data_cols:
        verschilt |= merged[f"{col}_src"].astype(str) != merged[f"{col}_dwh"].astype(str)

    gewijzigd = df_src[df_src[pk].isin(merged.loc[verschilt, pk])].copy()
    return nieuwe, gewijzigd


def load_scd1(df_src, dwh_conn, tabel, pk):
    logger.info(f"DWH_ETL|Start SCD1: {tabel} ({len(df_src)} rows)")
    sql_select = f"SELECT * FROM {tabel}"
    logger.info(f"DWH_ETL|SQL: {sql_select}")
    df_dwh = pd.read_sql_query(sql_select, dwh_conn)
    nieuwe, gewijzigd = vergelijk(df_src, df_dwh, pk)

    if not nieuwe.empty:
        logger.info(f"DWH_ETL|SQL: INSERT INTO {tabel} (...) VALUES (...) [{len(nieuwe)} rows]")
        nieuwe.to_sql(tabel, dwh_conn, if_exists="append", index=False)
    else:
        logger.info(f"DWH_ETL|SQL: INSERT INTO {tabel} (...) VALUES (...) [0 rows]")

    for _, row in gewijzigd.iterrows():
        cols = [c for c in df_src.columns if c != pk]
        sets = ", ".join(f"{c} = ?" for c in cols)
        sql_update = f"UPDATE {tabel} SET {sets} WHERE {pk} = ?"
        logger.info(f"DWH_ETL|SQL: {sql_update}")
        dwh_conn.execute(
            sql_update,
            [row[c] for c in cols] + [row[pk]],
        )

    dwh_conn.commit()
    logger.success(f"DWH_ETL|SCD1 done: {tabel} (new={len(nieuwe)}, changed={len(gewijzigd)})")


def load_scd2(df_src, dwh_conn, tabel, bk):
    logger.info(f"DWH_ETL|Start SCD2: {tabel} ({len(df_src)} rows)")
    sk = f"{tabel.lower()}_sk"
    sql_select = f"SELECT * FROM {tabel} WHERE eindtijd IS NULL"
    logger.info(f"DWH_ETL|SQL: {sql_select}")
    df_dwh = pd.read_sql_query(sql_select, dwh_conn)
    src_cols = list(df_src.columns)
    df_dwh_cmp = df_dwh[src_cols] if not df_dwh.empty else pd.DataFrame(columns=src_cols)
    nieuwe, gewijzigd = vergelijk(df_src, df_dwh_cmp, bk)

    if not nieuwe.empty:
        logger.info(f"DWH_ETL|SQL: INSERT INTO {tabel} (...) VALUES (...) [{len(nieuwe)} rows]")
        ins = nieuwe.assign(begintijd=VANDAAG, eindtijd=None)
        ins.to_sql(tabel, dwh_conn, if_exists="append", index=False)
    else:
        logger.info(f"DWH_ETL|SQL: INSERT INTO {tabel} (...) VALUES (...) [0 rows]")

    for _, row in gewijzigd.iterrows():
        old_sk = int(df_dwh.loc[df_dwh[bk] == row[bk], sk].iloc[0])
        sql_close = f"UPDATE {tabel} SET eindtijd = ? WHERE {sk} = ?"
        logger.info(f"DWH_ETL|SQL: {sql_close}")
        dwh_conn.execute(sql_close, [VANDAAG, old_sk])

        logger.info(f"DWH_ETL|SQL: INSERT INTO {tabel} (...) VALUES (...) [1 row]")
        new_row = {**row.to_dict(), "begintijd": VANDAAG, "eindtijd": None}
        pd.DataFrame([new_row]).to_sql(tabel, dwh_conn, if_exists="append", index=False)

    dwh_conn.commit()
    logger.success(f"DWH_ETL|SCD2 done: {tabel} (new={len(nieuwe)}, changed={len(gewijzigd)})")


def prepare_fact_keys(df, dwh_conn, tabel):
    # Gebruik surrogate keys alleen als de feit- en dimensietabellen die echt hebben.
    fact_cols = get_table_columns(dwh_conn, tabel)

    if "klant_sk" in fact_cols and "klantnr" in df.columns:
        sql_klant = "SELECT klantnr, klant_sk FROM Klant WHERE eindtijd IS NULL"
        logger.info(f"DWH_ETL|SQL: {sql_klant}")
        klant = pd.read_sql_query(sql_klant, dwh_conn)
        df = df.merge(klant, on="klantnr").drop(columns="klantnr")

    if "monteur_sk" in fact_cols and "monteurnr" in df.columns:
        sql_monteur = "SELECT monteurnr, monteur_sk FROM Monteur WHERE eindtijd IS NULL"
        logger.info(f"DWH_ETL|SQL: {sql_monteur}")
        monteur = pd.read_sql_query(sql_monteur, dwh_conn)
        df = df.merge(monteur, on="monteurnr").drop(columns="monteurnr")

    return df


def load_feit(df, dwh_conn, tabel, pk):
    logger.info(f"DWH_ETL|Start fact load: {tabel} ({len(df)} rows)")
    sql_select = f"SELECT {pk} FROM {tabel}"
    logger.info(f"DWH_ETL|SQL: {sql_select}")
    bestaand = pd.read_sql_query(sql_select, dwh_conn)[pk]
    nieuwe = df[~df[pk].isin(bestaand)]
    if not nieuwe.empty:
        logger.info(f"DWH_ETL|SQL: INSERT INTO {tabel} (...) VALUES (...) [{len(nieuwe)} rows]")
        nieuwe.to_sql(tabel, dwh_conn, if_exists="append", index=False)
    else:
        logger.info(f"DWH_ETL|SQL: INSERT INTO {tabel} (...) VALUES (...) [0 rows]")
    dwh_conn.commit()
    logger.success(f"DWH_ETL|Fact done: {tabel} (new={len(nieuwe)})")


logger.success("DWH_ETL|Functions ready")

2026-04-22 00:39:03|SUCCESS|DWH_ETL|Functions ready


In [36]:
# 4. ETL UITVOEREN

logger.info("DWH_ETL|Start ETL pipeline")

try:
    with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
        dwh.execute("PRAGMA foreign_keys = ON")

        logger.info("DWH_ETL|Start step: SCD1 dimensions")
        for tabel, pk, sql in SCD1_DIMS:
            df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=pk)
            df = afleiden(df, tabel)
            load_scd1(df, dwh, tabel, pk)
        logger.success("DWH_ETL|Step done: SCD1 dimensions")

        if SCD2_DIMS:
            logger.info("DWH_ETL|Start step: SCD2 dimensions")
            for tabel, bk, sql in SCD2_DIMS:
                df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=bk)
                df = afleiden(df, tabel)
                load_scd2(df, dwh, tabel, bk)
            logger.success("DWH_ETL|Step done: SCD2 dimensions")
        else:
            logger.info("DWH_ETL|Skip step: SCD2 dimensions")

        logger.info("DWH_ETL|Start step: fact tables")
        for tabel, pk, sql in FEITEN:
            df = pd.read_sql_query(sql, sdm)

            if tabel in ("Accessoire_Verkoop", "Fiets_Verkoop"):
                df["omzet"] = df["aantal"] * df["verkoopprijs"]
                df["inkoopwaarde"] = df["aantal"] * df["inkoopprijs"]
                df["brutowinst"] = df["omzet"] - df["inkoopwaarde"]

            elif tabel == "Onderhoud":
                start = pd.to_timedelta(df["starttijd"].astype(str))
                eind = pd.to_timedelta(df["eindtijd"].astype(str))
                df["duur_minuten"] = ((eind - start).dt.total_seconds() / 60).astype(int)
                df["arbeidskosten"] = (df["duur_minuten"] / 60 * df["uurloon"]).round(2)

            df = prepare_fact_keys(df, dwh, tabel)
            load_feit(df, dwh, tabel, pk)
        logger.success("DWH_ETL|Step done: fact tables")

    logger.success("DWH_ETL|ETL pipeline done")

except Exception as e:
    logger.exception(f"DWH_ETL|ETL pipeline failed ({e})")

2026-04-22 00:39:07|INFO|DWH_ETL|Start ETL pipeline
2026-04-22 00:39:07|INFO|DWH_ETL|Start step: SCD1 dimensions
2026-04-22 00:39:07|INFO|DWH_ETL|Start SCD1: Datum (196 rows)
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: SELECT * FROM Datum
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: INSERT INTO Datum (...) VALUES (...) [0 rows]
2026-04-22 00:39:07|SUCCESS|DWH_ETL|SCD1 done: Datum (new=0, changed=0)
2026-04-22 00:39:07|INFO|DWH_ETL|Start SCD1: Filiaal (5 rows)
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: SELECT * FROM Filiaal
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: INSERT INTO Filiaal (...) VALUES (...) [0 rows]
2026-04-22 00:39:07|SUCCESS|DWH_ETL|SCD1 done: Filiaal (new=0, changed=0)
2026-04-22 00:39:07|INFO|DWH_ETL|Start SCD1: Leverancier (5 rows)
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: SELECT * FROM Leverancier
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: INSERT INTO Leverancier (...) VALUES (...) [0 rows]
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: UPDATE Leverancier SET naam = ?, adres = ?, woonplaats = ? WHERE levera

2026-04-22 00:39:07|INFO|DWH_ETL|Start SCD1: Fabrikant (11 rows)
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: SELECT * FROM Fabrikant
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: INSERT INTO Fabrikant (...) VALUES (...) [0 rows]
2026-04-22 00:39:07|SUCCESS|DWH_ETL|SCD1 done: Fabrikant (new=0, changed=0)
2026-04-22 00:39:07|INFO|DWH_ETL|Start SCD1: Fiets (75 rows)
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: SELECT * FROM Fiets
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: INSERT INTO Fiets (...) VALUES (...) [0 rows]
2026-04-22 00:39:07|SUCCESS|DWH_ETL|SCD1 done: Fiets (new=0, changed=0)
2026-04-22 00:39:07|SUCCESS|DWH_ETL|Step done: SCD1 dimensions
2026-04-22 00:39:07|INFO|DWH_ETL|Start step: SCD2 dimensions
2026-04-22 00:39:07|INFO|DWH_ETL|Start SCD2: Klant (25 rows)
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: SELECT * FROM Klant WHERE eindtijd IS NULL
2026-04-22 00:39:07|INFO|DWH_ETL|SQL: INSERT INTO Klant (...) VALUES (...) [0 rows]
2026-04-22 00:39:07|SUCCESS|DWH_ETL|SCD2 done: Klant (new=0, changed=0)
2026-04-22 0

## Reset DWH

In [ ]:
# Reset DWH (alle tabellen leegmaken)

logger.info("DWH_ETL|Start reset")

try:
    with sqlite3.connect(DB_DWH) as dwh:
        for tabel in reversed(ALLE_TABELLEN):
            sql_delete = f"DELETE FROM {tabel}"
            try:
                logger.info(f"DWH_ETL|SQL: {sql_delete}")
                dwh.execute(sql_delete)
            except sqlite3.Error as e:
                logger.error(f"DWH_ETL|SQL failed: {sql_delete} ({e})")
        dwh.commit()
    logger.success("DWH_ETL|Reset done")
except Exception as e:
    logger.exception(f"DWH_ETL|Reset failed ({e})")

2026-04-20 23:26:55.247 | INFO     | __main__:<module>:3 - START RESET DWH
2026-04-20 23:26:55.254 | INFO     | __main__:<module>:10 -   ✓ reset: Onderhoud
2026-04-20 23:26:55.255 | INFO     | __main__:<module>:10 -   ✓ reset: Fiets_Verkoop
2026-04-20 23:26:55.257 | INFO     | __main__:<module>:10 -   ✓ reset: Accessoire_Verkoop
2026-04-20 23:26:55.258 | INFO     | __main__:<module>:10 -   ✓ reset: Monteur
2026-04-20 23:26:55.259 | INFO     | __main__:<module>:10 -   ✓ reset: Klant
2026-04-20 23:26:55.260 | INFO     | __main__:<module>:10 -   ✓ reset: Fiets
2026-04-20 23:26:55.261 | INFO     | __main__:<module>:10 -   ✓ reset: Fabrikant
2026-04-20 23:26:55.262 | INFO     | __main__:<module>:10 -   ✓ reset: Accessoire
2026-04-20 23:26:55.264 | INFO     | __main__:<module>:10 -   ✓ reset: Leverancier
2026-04-20 23:26:55.265 | INFO     | __main__:<module>:10 -   ✓ reset: Filiaal
2026-04-20 23:26:55.266 | INFO     | __main__:<module>:10 -   ✓ reset: Datum
2026-04-20 23:26:55.269 | INFO    

## Verificatie

In [ ]:
# 5. VERIFICATIE — Aantal rijen per tabel
logger.info("DWH_ETL|Start verification")

with sqlite3.connect(DB_DWH) as dwh:
    rows = []
    for tabel in ALLE_TABELLEN:
        try:
            count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {tabel}", dwh).iloc[0, 0]
            rows.append({"Tabel": tabel, "Rijen": count, "Status": "✓" if count > 0 else "⚠ leeg"})
        except Exception as e:
            rows.append({"Tabel": tabel, "Rijen": "–", "Status": f"✗ {e}"})

logger.success(f"DWH_ETL|Verification done: {len(rows)} tables")
pd.DataFrame(rows)

2026-04-21 01:55:33.222 | INFO     | __main__:<module>:2 - Verificatie gestart...


,Tabel,Rijen,Status
0,Datum,196,✓
1,Filiaal,5,✓
2,Leverancier,5,✓
3,Accessoire,13,✓
4,Fabrikant,11,✓
5,Fiets,75,✓
6,Klant,29,✓
7,Monteur,15,✓
8,Accessoire_Verkoop,100,✓
9,Fiets_Verkoop,150,✓


In [ ]:
# 6. PREVIEW — Bekijk eerste rijen van een DWH tabel
PREVIEW_TABLE = "Klant"

with sqlite3.connect(DB_DWH) as dwh:
    df_preview = pd.read_sql_query(f"SELECT * FROM {PREVIEW_TABLE} LIMIT 10", dwh)

logger.success(f"DWH_ETL|Preview ready: {PREVIEW_TABLE} ({len(df_preview)} rows)")
df_preview

2026-04-20 23:26:55.311 | INFO     | __main__:<module>:7 - Preview 'Klant' — 0 rijen


,klant_sk,klantnr,naam,adres,woonplaats,geslacht,geboortedatum,leeftijdsklasse,begintijd,eindtijd


## Test 1 — Toevoegen (nieuwe rij)

Voeg een nieuwe Filiaal toe aan het SDM → herlaad ETL → controleer dat het in DWH verschijnt.

In [8]:
# TEST 1: Nieuwe rij toevoegen (SCD1 — Filiaal)
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    # VOOR — Filiaal 999 bestaat niet
    print("VOOR:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 999", dwh).to_string())

    # Toevoegen in SDM
    sdm.execute("INSERT INTO Accessoire_Verkoop_Filiaal VALUES (999, 'Test Filiaal', 'Teststraat 1', 'Zuid-Holland')")
    sdm.commit()

    # Herlaad SCD1
    df = pd.read_sql_query(SCD1_DIMS[1][2], sdm).drop_duplicates(subset="filiaalnr")
    load_scd1(afleiden(df, "Filiaal"), dwh, "Filiaal", "filiaalnr")

    # NA — Filiaal 999 moet nu bestaan
    print("NA:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 999", dwh).to_string())

    # Opruimen
    sdm.execute("DELETE FROM Accessoire_Verkoop_Filiaal WHERE filiaalnr = 999"); sdm.commit()
    dwh.execute("DELETE FROM Filiaal WHERE filiaalnr = 999"); dwh.commit()
    logger.info("Opgeruimd")

2026-04-20 23:26:55.340 | INFO     | __main__:load_scd1:75 -   ✓ Filiaal: 6 nieuw, 0 gewijzigd
2026-04-20 23:26:55.347 | INFO     | __main__:<module>:22 - Opgeruimd


VOOR: Empty DataFrame
Columns: [filiaalnr, naam, adres, provincie]
Index: []
NA:    filiaalnr          naam         adres     provincie
0        999  Test Filiaal  Teststraat 1  Zuid-Holland


## Test 2 — Wijzigen SCD Type 1 (Update/Overschrijven)

Wijzig een Filiaal-naam in het SDM → herlaad ETL → controleer dat de oude rij is **overschreven**.

In [9]:
# TEST 2: Wijzigen SCD1 (Filiaal naam overschrijven)
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    # VOOR
    print("VOOR:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

    # Bewaar origineel en wijzig SDM
    origineel = pd.read_sql_query("SELECT naam FROM Accessoire_Verkoop_Filiaal WHERE filiaalnr = 1", sdm).iloc[0, 0]
    sdm.execute("UPDATE Accessoire_Verkoop_Filiaal SET naam = 'TEST_SCD1' WHERE filiaalnr = 1"); sdm.commit()

    # Herlaad SCD1
    df = pd.read_sql_query(SCD1_DIMS[1][2], sdm).drop_duplicates(subset="filiaalnr")
    load_scd1(afleiden(df, "Filiaal"), dwh, "Filiaal", "filiaalnr")

    # NA — naam moet overschreven zijn
    print("NA:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

    # Herstel origineel
    sdm.execute("UPDATE Accessoire_Verkoop_Filiaal SET naam = ? WHERE filiaalnr = 1", [origineel]); sdm.commit()
    df = pd.read_sql_query(SCD1_DIMS[1][2], sdm).drop_duplicates(subset="filiaalnr")
    load_scd1(afleiden(df, "Filiaal"), dwh, "Filiaal", "filiaalnr")
    logger.info("Hersteld naar origineel")

2026-04-20 23:26:55.370 | INFO     | __main__:load_scd1:75 -   ✓ Filiaal: 0 nieuw, 0 gewijzigd
2026-04-20 23:26:55.379 | INFO     | __main__:load_scd1:75 -   ✓ Filiaal: 0 nieuw, 0 gewijzigd
2026-04-20 23:26:55.380 | INFO     | __main__:<module>:23 - Hersteld naar origineel


VOOR:    filiaalnr                 naam              adres      provincie
0          1  BikeWorld Amsterdam  Prinsengracht 100  Noord-Holland
NA:    filiaalnr                 naam              adres      provincie
0          1  BikeWorld Amsterdam  Prinsengracht 100  Noord-Holland


## Test 3 — Wijzigen klantdimensie

Dit testblok past zich aan aan de actieve DWH-strategie:
- Bij **SCD Type 2** krijgt de oude versie een `eindtijd` en komt er een nieuwe actuele versie bij
- Bij **SCD Type 1** wordt de bestaande klant gewoon overschreven

In [10]:
# TEST 3: Wijzigen klantdimensie volgens actief DWH-schema
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    klant_sql = SCD2_DIMS[0][2] if DWH_STRATEGIE == "scd2" else OPTIONELE_SCD1_DIMS[0][2]
    dwh_klant_sql = (
        "SELECT * FROM Klant WHERE klantnr = 1 ORDER BY klant_sk"
        if DWH_STRATEGIE == "scd2"
        else "SELECT * FROM Klant WHERE klantnr = 1"
    )

    df_klant = pd.read_sql_query(klant_sql, sdm).drop_duplicates(subset="klantnr")
    df_klant = afleiden(df_klant, "Klant")

    df_voor = pd.read_sql_query(dwh_klant_sql, dwh)
    if df_voor.empty:
        logger.info("Klant 1 niet aanwezig in DWH — initialiseer eerst de klantdimensie voor deze test")
        if DWH_STRATEGIE == "scd2":
            load_scd2(df_klant, dwh, "Klant", "klantnr")
        else:
            load_scd1(df_klant, dwh, "Klant", "klantnr")
        df_voor = pd.read_sql_query(dwh_klant_sql, dwh)

    if df_voor.empty:
        raise RuntimeError("Klant 1 bestaat niet in SDM of kon niet in DWH worden geladen.")

    print("VOOR:", df_voor.to_string())
    ORIGINELE_KLANT_WOONPLAATS = df_voor.iloc[0]["woonplaats"]

    bron_tabellen = ["Accessoire_Verkoop_Klant", "Fiets_Verkoop_Klant"]
    for bron_tabel in bron_tabellen:
        sdm.execute(
            f"UPDATE {bron_tabel} SET woonplaats = 'TEST_KLANT' WHERE klantnr = 1"
        )
    sdm.commit()

    df_klant = pd.read_sql_query(klant_sql, sdm).drop_duplicates(subset="klantnr")
    df_klant = afleiden(df_klant, "Klant")

    if DWH_STRATEGIE == "scd2":
        load_scd2(df_klant, dwh, "Klant", "klantnr")
        print(
            "NA:",
            pd.read_sql_query(dwh_klant_sql, dwh).to_string(),
        )
        logger.info("SCD2 gedetecteerd — oude versie afgesloten en nieuwe versie toegevoegd")
    else:
        load_scd1(df_klant, dwh, "Klant", "klantnr")
        print(
            "NA:",
            pd.read_sql_query(dwh_klant_sql, dwh).to_string(),
        )
        logger.info("SCD1 gedetecteerd — klant is overschreven in het DWH")

    logger.info("Brondata is aangepast voor de vervolgtest met een nieuw feit")

2026-04-20 23:26:55.396 | INFO     | __main__:<module>:17 - Klant 1 niet aanwezig in DWH — initialiseer eerst de klantdimensie voor deze test
2026-04-20 23:26:55.402 | INFO     | __main__:load_scd2:97 -   ✓ Klant: 25 nieuw, 0 gewijzigd (SCD2)
2026-04-20 23:26:55.421 | INFO     | __main__:load_scd2:97 -   ✓ Klant: 0 nieuw, 1 gewijzigd (SCD2)
2026-04-20 23:26:55.425 | INFO     | __main__:<module>:46 - SCD2 gedetecteerd — oude versie afgesloten en nieuwe versie toegevoegd
2026-04-20 23:26:55.426 | INFO     | __main__:<module>:55 - Brondata is aangepast voor de vervolgtest met een nieuw feit


VOOR:    klant_sk  klantnr        naam          adres woonplaats geslacht geboortedatum leeftijdsklasse   begintijd eindtijd
0        54        1  Jan Jansen  Kerkstraat 12  Amsterdam        M    1985-03-22      Middelbaar  2026-04-20     None
NA:    klant_sk  klantnr        naam          adres  woonplaats geslacht geboortedatum leeftijdsklasse   begintijd    eindtijd
0        54        1  Jan Jansen  Kerkstraat 12   Amsterdam        M    1985-03-22      Middelbaar  2026-04-20  2026-04-20
1        79        1  Jan Jansen  Kerkstraat 12  TEST_KLANT        M    1985-03-22      Middelbaar  2026-04-20         NaN


## Test 4 — Verwijderen + herstel via ETL

Verwijder een Filiaal uit het DWH → herlaad ETL → controleer dat de rij **terugkomt** vanuit het SDM.

In [11]:
# TEST 4: Verwijderen uit DWH en herstel via ETL
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    # VOOR
    print("VOOR:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

    # Verwijder uit DWH (FK tijdelijk uit voor test-doeleinden)
    dwh.execute("PRAGMA foreign_keys = OFF")
    dwh.execute("DELETE FROM Filiaal WHERE filiaalnr = 1"); dwh.commit()
    dwh.execute("PRAGMA foreign_keys = ON")
    print("NA DELETE:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

    # Herlaad SCD1 — filiaal komt terug vanuit SDM
    df = pd.read_sql_query(SCD1_DIMS[1][2], sdm).drop_duplicates(subset="filiaalnr")
    load_scd1(afleiden(df, "Filiaal"), dwh, "Filiaal", "filiaalnr")

    # Hersteld
    print("NA ETL:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

2026-04-20 23:26:55.451 | INFO     | __main__:load_scd1:75 -   ✓ Filiaal: 1 nieuw, 0 gewijzigd


VOOR:    filiaalnr                 naam              adres      provincie
0          1  BikeWorld Amsterdam  Prinsengracht 100  Noord-Holland
NA DELETE: Empty DataFrame
Columns: [filiaalnr, naam, adres, provincie]
Index: []
NA ETL:    filiaalnr                 naam              adres      provincie
0          1  BikeWorld Amsterdam  Prinsengracht 100  Noord-Holland


## Test 5 — Nieuw feit met juiste sleutel

Dit testblok controleert welke sleutel het feit krijgt op basis van het actieve schema:
- Bij **SCD Type 2** moet het feit de actuele surrogate key (`klant_sk`) krijgen
- Bij **SCD Type 1** blijft het feit de natural key (`klantnr`) gebruiken

In [12]:
# TEST 5: Nieuw feit krijgt de juiste klant-sleutel voor het actieve DWH-schema
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    if "ORIGINELE_KLANT_WOONPLAATS" not in globals():
        raise RuntimeError("Draai eerst Test 3 zodat de originele klantwaarde bekend is.")

    if pd.read_sql_query("SELECT COUNT(*) AS n FROM Datum", dwh).iloc[0, 0] == 0:
        logger.info("DWH is leeg — laad eerst de benodigde dimensies voor deze test")
        for tabel, pk, sql in SCD1_DIMS:
            df_dim = pd.read_sql_query(sql, sdm).drop_duplicates(subset=pk)
            df_dim = afleiden(df_dim, tabel)
            load_scd1(df_dim, dwh, tabel, pk)

        for tabel, bk, sql in SCD2_DIMS:
            df_dim = pd.read_sql_query(sql, sdm).drop_duplicates(subset=bk)
            df_dim = afleiden(df_dim, tabel)
            load_scd2(df_dim, dwh, tabel, bk)

    test_nr = None
    try:
        if DWH_STRATEGIE == "scd2":
            print(
                "Klant versies:",
                pd.read_sql_query(
                    "SELECT klant_sk, klantnr, woonplaats, begintijd, eindtijd FROM Klant WHERE klantnr = 1 ORDER BY klant_sk",
                    dwh,
                ).to_string(),
            )
        else:
            print(
                "Klant rij:",
                pd.read_sql_query("SELECT * FROM Klant WHERE klantnr = 1", dwh).to_string(),
            )

        test_datum = pd.read_sql_query(
            "SELECT datum FROM Datum ORDER BY datum LIMIT 1",
            dwh,
        ).iloc[0, 0]
        max_nr = pd.read_sql_query(
            "SELECT COALESCE(MAX(accessoire_verkoopnr), 0) AS m FROM Accessoire_Verkoop",
            sdm,
        ).iloc[0, 0]
        test_nr = int(max_nr) + 1
        sdm.execute(
            "INSERT INTO Accessoire_Verkoop VALUES (?, ?, 2, 15.00, 1, 1, 1)",
            [test_nr, test_datum],
        )
        sdm.commit()

        df = pd.read_sql_query(FEITEN[0][2], sdm)
        df["omzet"] = df["aantal"] * df["verkoopprijs"]
        df["inkoopwaarde"] = df["aantal"] * df["inkoopprijs"]
        df["brutowinst"] = df["omzet"] - df["inkoopwaarde"]
        df = prepare_fact_keys(df, dwh, "Accessoire_Verkoop")
        load_feit(df, dwh, "Accessoire_Verkoop", "accessoire_verkoopnr")

        print(
            "Feit:",
            pd.read_sql_query(
                f"SELECT * FROM Accessoire_Verkoop WHERE accessoire_verkoopnr = {test_nr}",
                dwh,
            ).to_string(),
        )
    finally:
        if test_nr is not None:
            sdm.execute(
                "DELETE FROM Accessoire_Verkoop WHERE accessoire_verkoopnr = ?",
                [test_nr],
            )
            dwh.execute(
                "DELETE FROM Accessoire_Verkoop WHERE accessoire_verkoopnr = ?",
                [test_nr],
            )
            sdm.commit()
            dwh.commit()

        bron_tabellen = ["Accessoire_Verkoop_Klant", "Fiets_Verkoop_Klant"]
        for bron_tabel in bron_tabellen:
            sdm.execute(
                f"UPDATE {bron_tabel} SET woonplaats = ? WHERE klantnr = 1",
                [ORIGINELE_KLANT_WOONPLAATS],
            )
        sdm.commit()

        klant_sql = SCD2_DIMS[0][2] if DWH_STRATEGIE == "scd2" else OPTIONELE_SCD1_DIMS[0][2]
        df_klant = pd.read_sql_query(klant_sql, sdm).drop_duplicates(subset="klantnr")
        df_klant = afleiden(df_klant, "Klant")

        if DWH_STRATEGIE == "scd2":
            load_scd2(df_klant, dwh, "Klant", "klantnr")
            logger.info("Opgeruimd + klant hersteld via SCD2")
        else:
            load_scd1(df_klant, dwh, "Klant", "klantnr")
            logger.info("Opgeruimd + klant hersteld via SCD1")

2026-04-20 23:26:55.467 | INFO     | __main__:<module>:9 - DWH is leeg — laad eerst de benodigde dimensies voor deze test
2026-04-20 23:26:55.475 | INFO     | __main__:load_scd1:75 -   ✓ Datum: 196 nieuw, 0 gewijzigd
2026-04-20 23:26:55.482 | INFO     | __main__:load_scd1:75 -   ✓ Filiaal: 0 nieuw, 0 gewijzigd
2026-04-20 23:26:55.487 | INFO     | __main__:load_scd1:75 -   ✓ Leverancier: 5 nieuw, 0 gewijzigd
2026-04-20 23:26:55.492 | INFO     | __main__:load_scd1:75 -   ✓ Accessoire: 13 nieuw, 0 gewijzigd
2026-04-20 23:26:55.497 | INFO     | __main__:load_scd1:75 -   ✓ Fabrikant: 11 nieuw, 0 gewijzigd
2026-04-20 23:26:55.504 | INFO     | __main__:load_scd1:75 -   ✓ Fiets: 75 nieuw, 0 gewijzigd
2026-04-20 23:26:55.516 | INFO     | __main__:load_scd2:97 -   ✓ Klant: 0 nieuw, 0 gewijzigd (SCD2)
2026-04-20 23:26:55.522 | INFO     | __main__:load_scd2:97 -   ✓ Monteur: 15 nieuw, 0 gewijzigd (SCD2)


Klant versies:    klant_sk  klantnr  woonplaats   begintijd    eindtijd
0        54        1   Amsterdam  2026-04-20  2026-04-20
1        79        1  TEST_KLANT  2026-04-20         NaN


2026-04-20 23:26:55.536 | INFO     | __main__:load_feit:128 -   ✓ Accessoire_Verkoop: 101 nieuwe rijen
2026-04-20 23:26:55.559 | INFO     | __main__:load_scd2:97 -   ✓ Klant: 0 nieuw, 1 gewijzigd (SCD2)
2026-04-20 23:26:55.560 | INFO     | __main__:<module>:92 - Opgeruimd + klant hersteld via SCD2


Feit:    accessoire_verkoopnr       datum  klant_sk  monteur_sk  filiaalnr  accessoirenr  leveranciernr  aantal  standaardprijs  verkoopprijs  inkoopprijs  omzet  inkoopwaarde  brutowinst
0                   101  2024-01-02        79          33          1             1              1       2           14.99          15.0          8.5   30.0          17.0        13.0


## Test 6 — Herlaad DWH na SDM-wijziging

Simulatie: wijzig data in het SDM en draai de **volledige ETL** opnieuw. Controleer dat:
- **SCD Type 1** dimensie (Leverancier) de wijziging **overschrijft**
- **SCD Type 2** dimensie (Klant) — afhankelijk van het actieve schema — correct wordt bijgewerkt
- Bestaande feiten **ongewijzigd** blijven (geen duplicaten)

In [37]:
# TEST 6: Herlaad DWH na SDM-wijziging — volledige ETL-loop
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    # --- Stap 1: Bewaar originelen ---
    orig_lev = pd.read_sql_query(
        "SELECT naam FROM Accessoire_Verkoop_Leverancier WHERE leveranciernr = 1", sdm
    ).iloc[0, 0]
    orig_klant = pd.read_sql_query(
        "SELECT woonplaats FROM Accessoire_Verkoop_Klant WHERE klantnr = 1", sdm
    ).iloc[0, 0]

    # Bewaar feit-aantallen VOOR de herlaad
    av_count_voor = pd.read_sql_query("SELECT COUNT(*) AS n FROM Accessoire_Verkoop", dwh).iloc[0, 0]
    fv_count_voor = pd.read_sql_query("SELECT COUNT(*) AS n FROM Fiets_Verkoop", dwh).iloc[0, 0]
    oh_count_voor = pd.read_sql_query("SELECT COUNT(*) AS n FROM Onderhoud", dwh).iloc[0, 0]

    print("VOOR HERLAAD")
    print("Leverancier 1:", pd.read_sql_query("SELECT * FROM Leverancier WHERE leveranciernr = 1", dwh).to_string())
    if DWH_STRATEGIE == "scd2":
        print("Klant 1:", pd.read_sql_query(
            "SELECT klant_sk, klantnr, woonplaats, begintijd, eindtijd FROM Klant WHERE klantnr = 1 ORDER BY klant_sk", dwh
        ).to_string())
    else:
        print("Klant 1:", pd.read_sql_query("SELECT * FROM Klant WHERE klantnr = 1", dwh).to_string())

    # --- Stap 2: Wijzig SDM brondata ---
    # SCD1: Leverancier naam wijzigen (in ALLE brontabellen)
    for tbl in ["Accessoire_Verkoop_Leverancier", "Accessoire_Inkoop_Leverancier"]:
        sql_update_lev = f"UPDATE {tbl} SET naam = 'TEST_HERLAAD' WHERE leveranciernr = 1"
        logger.info(f"DWH_ETL|SQL: {sql_update_lev}")
        sdm.execute(sql_update_lev)

    # SCD1/SCD2: Klant woonplaats wijzigen (in ALLE brontabellen)
    for tbl in ["Accessoire_Verkoop_Klant", "Fiets_Verkoop_Klant"]:
        sql_update_klant = f"UPDATE {tbl} SET woonplaats = 'HERLAAD_STAD' WHERE klantnr = 1"
        logger.info(f"DWH_ETL|SQL: {sql_update_klant}")
        sdm.execute(sql_update_klant)
    sdm.commit()
    logger.success("DWH_ETL|Source update done")

    # --- Stap 3: Draai de volledige ETL opnieuw ---
    logger.info("DWH_ETL|Start reload")
    for tabel, pk, sql in SCD1_DIMS:
        df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=pk)
        df = afleiden(df, tabel)
        load_scd1(df, dwh, tabel, pk)

    for tabel, bk, sql in SCD2_DIMS:
        df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=bk)
        df = afleiden(df, tabel)
        load_scd2(df, dwh, tabel, bk)

    for tabel, pk, sql in FEITEN:
        df = pd.read_sql_query(sql, sdm)
        if tabel in ("Accessoire_Verkoop", "Fiets_Verkoop"):
            df["omzet"] = df["aantal"] * df["verkoopprijs"]
            df["inkoopwaarde"] = df["aantal"] * df["inkoopprijs"]
            df["brutowinst"] = df["omzet"] - df["inkoopwaarde"]
        elif tabel == "Onderhoud":
            start = pd.to_timedelta(df["starttijd"].astype(str))
            eind = pd.to_timedelta(df["eindtijd"].astype(str))
            df["duur_minuten"] = ((eind - start).dt.total_seconds() / 60).astype(int)
            df["arbeidskosten"] = (df["duur_minuten"] / 60 * df["uurloon"]).round(2)
        df = prepare_fact_keys(df, dwh, tabel)
        load_feit(df, dwh, tabel, pk)
    logger.success("DWH_ETL|Reload done")

    # --- Stap 4: Controleer resultaten ---
    print("\nNA HERLAAD")

    # SCD1 check: Leverancier moet overschreven zijn
    lev_na = pd.read_sql_query("SELECT * FROM Leverancier WHERE leveranciernr = 1", dwh)
    print("Leverancier 1:", lev_na.to_string())
    assert lev_na.iloc[0]["naam"] == "TEST_HERLAAD", "SCD1 FOUT: Leverancier niet overschreven!"
    logger.success("DWH_ETL|Check done: leverancier SCD1")

    # SCD2/SCD1 check: Klant
    if DWH_STRATEGIE == "scd2":
        klant_na = pd.read_sql_query(
            "SELECT klant_sk, klantnr, woonplaats, begintijd, eindtijd FROM Klant WHERE klantnr = 1 ORDER BY klant_sk", dwh
        )
        print("Klant 1:", klant_na.to_string())
        actueel = klant_na[klant_na["eindtijd"].isna()]
        assert len(actueel) == 1, "SCD2 FOUT: verwacht precies 1 actuele klant-versie!"
        assert actueel.iloc[0]["woonplaats"] == "HERLAAD_STAD", "SCD2 FOUT: woonplaats niet bijgewerkt!"
        afgesloten = klant_na[klant_na["eindtijd"].notna()]
        assert len(afgesloten) >= 1, "SCD2 FOUT: oude versie niet afgesloten!"
        logger.success(f"DWH_ETL|Check done: klant SCD2 ({len(afgesloten)} old)")
    else:
        klant_na = pd.read_sql_query("SELECT * FROM Klant WHERE klantnr = 1", dwh)
        print("Klant 1:", klant_na.to_string())
        assert klant_na.iloc[0]["woonplaats"] == "HERLAAD_STAD", "SCD1 FOUT: Klant niet overschreven!"
        logger.success("DWH_ETL|Check done: klant SCD1")

    # Feit check: geen duplicaten — alleen controleren als de tabel al gevuld was
    av_count_na = pd.read_sql_query("SELECT COUNT(*) AS n FROM Accessoire_Verkoop", dwh).iloc[0, 0]
    fv_count_na = pd.read_sql_query("SELECT COUNT(*) AS n FROM Fiets_Verkoop", dwh).iloc[0, 0]
    oh_count_na = pd.read_sql_query("SELECT COUNT(*) AS n FROM Onderhoud", dwh).iloc[0, 0]
    print(f"\nFeit-aantallen: AV {av_count_voor}→{av_count_na}, FV {fv_count_voor}→{fv_count_na}, OH {oh_count_voor}→{oh_count_na}")

    if av_count_voor > 0:
        assert av_count_na == av_count_voor, f"FOUT: Accessoire_Verkoop duplicaten! {av_count_voor} → {av_count_na}"
    else:
        logger.info(f"DWH_ETL|Fact initialized: Accessoire_Verkoop ({av_count_na})")

    if fv_count_voor > 0:
        assert fv_count_na == fv_count_voor, f"FOUT: Fiets_Verkoop duplicaten! {fv_count_voor} → {fv_count_na}"
    else:
        logger.info(f"DWH_ETL|Fact initialized: Fiets_Verkoop ({fv_count_na})")

    if oh_count_voor > 0:
        assert oh_count_na == oh_count_voor, f"FOUT: Onderhoud duplicaten! {oh_count_voor} → {oh_count_na}"
    else:
        logger.info(f"DWH_ETL|Fact initialized: Onderhoud ({oh_count_na})")

    logger.success("DWH_ETL|Check done: no duplicates")

    # --- Stap 5: Herstel originele SDM-data ---
    for tbl in ["Accessoire_Verkoop_Leverancier", "Accessoire_Inkoop_Leverancier"]:
        sql_restore_lev = f"UPDATE {tbl} SET naam = ? WHERE leveranciernr = 1"
        logger.info(f"DWH_ETL|SQL: {sql_restore_lev}")
        sdm.execute(sql_restore_lev, [orig_lev])
    for tbl in ["Accessoire_Verkoop_Klant", "Fiets_Verkoop_Klant"]:
        sql_restore_klant = f"UPDATE {tbl} SET woonplaats = ? WHERE klantnr = 1"
        logger.info(f"DWH_ETL|SQL: {sql_restore_klant}")
        sdm.execute(sql_restore_klant, [orig_klant])
    sdm.commit()

    # Herstel DWH dimensies naar origineel
    for tabel, pk, sql in SCD1_DIMS:
        df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=pk)
        df = afleiden(df, tabel)
        load_scd1(df, dwh, tabel, pk)
    for tabel, bk, sql in SCD2_DIMS:
        df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=bk)
        df = afleiden(df, tabel)
        load_scd2(df, dwh, tabel, bk)

    logger.success("DWH_ETL|Restore done")
    logger.success("DWH_ETL|Test 6 passed")

2026-04-22 00:39:19|INFO|DWH_ETL|SQL: UPDATE Accessoire_Verkoop_Leverancier SET naam = 'TEST_HERLAAD' WHERE leveranciernr = 1
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: UPDATE Accessoire_Inkoop_Leverancier SET naam = 'TEST_HERLAAD' WHERE leveranciernr = 1
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: UPDATE Accessoire_Verkoop_Klant SET woonplaats = 'HERLAAD_STAD' WHERE klantnr = 1
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: UPDATE Fiets_Verkoop_Klant SET woonplaats = 'HERLAAD_STAD' WHERE klantnr = 1
2026-04-22 00:39:19|SUCCESS|DWH_ETL|Source update done
2026-04-22 00:39:19|INFO|DWH_ETL|Start reload
2026-04-22 00:39:19|INFO|DWH_ETL|Start SCD1: Datum (196 rows)
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: SELECT * FROM Datum
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: INSERT INTO Datum (...) VALUES (...) [0 rows]
2026-04-22 00:39:19|SUCCESS|DWH_ETL|SCD1 done: Datum (new=0, changed=0)
2026-04-22 00:39:19|INFO|DWH_ETL|Start SCD1: Filiaal (5 rows)


2026-04-22 00:39:19|INFO|DWH_ETL|SQL: SELECT * FROM Filiaal
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: INSERT INTO Filiaal (...) VALUES (...) [0 rows]
2026-04-22 00:39:19|SUCCESS|DWH_ETL|SCD1 done: Filiaal (new=0, changed=0)
2026-04-22 00:39:19|INFO|DWH_ETL|Start SCD1: Leverancier (5 rows)
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: SELECT * FROM Leverancier
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: INSERT INTO Leverancier (...) VALUES (...) [0 rows]
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: UPDATE Leverancier SET naam = ?, adres = ?, woonplaats = ? WHERE leveranciernr = ?
2026-04-22 00:39:19|SUCCESS|DWH_ETL|SCD1 done: Leverancier (new=0, changed=1)
2026-04-22 00:39:19|INFO|DWH_ETL|Start SCD1: Accessoire (13 rows)
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: SELECT * FROM Accessoire
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: INSERT INTO Accessoire (...) VALUES (...) [0 rows]
2026-04-22 00:39:19|SUCCESS|DWH_ETL|SCD1 done: Accessoire (new=0, changed=0)
2026-04-22 00:39:19|INFO|DWH_ETL|Start SCD1: Fabrikant (11 rows)
2

VOOR HERLAAD
Leverancier 1:    leveranciernr          naam             adres woonplaats
0              1  BikeParts BV  Fietsenstraat 12  Amsterdam
Klant 1:    klant_sk  klantnr    woonplaats   begintijd    eindtijd
0        83        1     Amsterdam  2026-04-21  2026-04-21
1       108        1    TEST_KLANT  2026-04-21  2026-04-21
2       109        1     Amsterdam  2026-04-21  2026-04-21
3       110        1  HERLAAD_STAD  2026-04-21  2026-04-21
4       111        1     Amsterdam  2026-04-21  2026-04-22
5       112        1  HERLAAD_STAD  2026-04-22  2026-04-22
6       113        1     Amsterdam  2026-04-22         NaN


2026-04-22 00:39:19|INFO|DWH_ETL|SQL: INSERT INTO Klant (...) VALUES (...) [0 rows]
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: UPDATE Klant SET eindtijd = ? WHERE klant_sk = ?
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: INSERT INTO Klant (...) VALUES (...) [1 row]
2026-04-22 00:39:19|SUCCESS|DWH_ETL|SCD2 done: Klant (new=0, changed=1)
2026-04-22 00:39:19|INFO|DWH_ETL|Start SCD2: Monteur (15 rows)
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: SELECT * FROM Monteur WHERE eindtijd IS NULL
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: INSERT INTO Monteur (...) VALUES (...) [0 rows]
2026-04-22 00:39:19|SUCCESS|DWH_ETL|SCD2 done: Monteur (new=0, changed=0)
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: SELECT klantnr, klant_sk FROM Klant WHERE eindtijd IS NULL
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: SELECT monteurnr, monteur_sk FROM Monteur WHERE eindtijd IS NULL
2026-04-22 00:39:19|INFO|DWH_ETL|Start fact load: Accessoire_Verkoop (100 rows)
2026-04-22 00:39:19|INFO|DWH_ETL|SQL: SELECT accessoire_verkoopnr FROM Accessoire_Verkoo


NA HERLAAD
Leverancier 1:    leveranciernr          naam             adres woonplaats
0              1  TEST_HERLAAD  Fietsenstraat 12  Amsterdam
Klant 1:    klant_sk  klantnr    woonplaats   begintijd    eindtijd
0        83        1     Amsterdam  2026-04-21  2026-04-21
1       108        1    TEST_KLANT  2026-04-21  2026-04-21
2       109        1     Amsterdam  2026-04-21  2026-04-21
3       110        1  HERLAAD_STAD  2026-04-21  2026-04-21
4       111        1     Amsterdam  2026-04-21  2026-04-22
5       112        1  HERLAAD_STAD  2026-04-22  2026-04-22
6       113        1     Amsterdam  2026-04-22  2026-04-22
7       114        1  HERLAAD_STAD  2026-04-22         NaN

Feit-aantallen: AV 100→100, FV 150→150, OH 50→50
